In [ ]:
'''
python version 3.10.12
'''

In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [1]:
import argparse
import sys
import os
import numpy as np
import torch
from torch import nn
from torch import Tensor
import yaml
from model import RawNet
from torch.nn import functional as F
import librosa
import json
from datetime import datetime
from torch.utils.data import DataLoader, Dataset
import time
from tqdm import tqdm

In [2]:
SAMPLE_RATE=16000
class Dataset_LibriSeVoc(Dataset):
    
    def __init__(self, dataset_path, protocol_file):
            self.dataset_path = dataset_path
            self.protocol_file=protocol_file
            
            y_list=[]
            path_list=[]
            audio_list=[]
            variant_list=[]
            random_token_list=[]
            src_list=[]
            with open (self.protocol_file, 'r') as f:
                audio_lines=f.readlines()
            for line in audio_lines:
                variant,audio_path,random_token,src,label=line.strip().split(' ')
                path_list.append(os.path.join(self.dataset_path,audio_path))
                variant_list.append(variant)
                random_token_list.append(random_token)
                audio_list.append(audio_path)
                src_list.append(src)
                y_list.append(0 if label=='bonafide' else 1)
            self.path_list=path_list
            self.y_list=y_list
            self.audio_list=audio_list
            self.variant_list=variant_list
            self.random_token_list=random_token_list
            self.src_list=src_list
            
            
            print('Load data from {}'.format(self.dataset_path))

    def __len__(self):
            
            return len(self.path_list)


    def __getitem__(self, index):
            self.cut=64600
            path = self.path_list[index]
            Y = self.y_list[index]
            X, fs = librosa.load(path, sr=None)
            X_pad = pad(X,self.cut)
            x_inp = Tensor(X_pad)
            y_inp = Y
            file_name=self.audio_list[index]
            variant_inp = self.variant_list[index]
            random_token_inp = self.random_token_list[index]
            src_inp = self.src_list[index]
            return x_inp, variant_inp,file_name,random_token_inp,src_inp, y_inp ==  0,
def pad(x, max_len=64600):
    x_len = x.shape[0]
    if x_len >= max_len:
        return x[:max_len]
    # need to pad
    num_repeats = int(max_len / x_len)+1
    padded_x = np.tile(x, (1, num_repeats))[:, :max_len][0]
    return padded_x	

def load_sample(sample_path, max_len = 64000):
    
    y_list = []
    y, sr = librosa.load(sample_path, sr=None)
    
    if sr != 16000:
        y = librosa.resample(y, orig_sr = sr, target_sr = 16000)
        
    if(len(y) <= max_len):
        return [Tensor(pad(y, max_len))]
        
    for i in range(int(len(y)/max_len)):
        if (i+1) ==  range(int(len(y)/max_len)):
            y_seg = y[i*max_len : ]
        else:
            y_seg = y[i*max_len : (i+1)*max_len]
        # print(len(y_seg))
        y_pad = pad(y_seg, max_len)
        y_inp = Tensor(y_pad)
        
        y_list.append(y_inp)
        
    return y_list

In [ ]:
def get_score(eval_path,data_path,save_path,comment:str=None):
    input_path=data_path
    model_path='change this to the RawNet2-Vocoder model path ' # download here [https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/Rawnet2-Vocoder.pth?download=true]
    dir_yaml = 'model_config_RawNet.yaml'
    with open(dir_yaml, 'r') as f_yaml:
        parser1 = yaml.safe_load(f_yaml)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Device: {}'.format(device))
    
    # init model
    model = RawNet(parser1['model'], device)
    model =(model).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    print('Model loaded : {}'.format(model_path))
    
    model.eval()
    # out_list_multi = []
    score_list_binary = []
    eval_Set=Dataset_LibriSeVoc(data_path,eval_path)
    eval_loader=DataLoader(eval_Set, batch_size=24, shuffle=False)
    variant_list=[]
    random_token_list=[]
    src_list=[]
    file_name_list=[]
    label_list=[]
    start_time = time.time()
    for step, (input, variant,file_name,random_token, src, target) in tqdm(enumerate(eval_loader)):
   
        m_batch = input.to(device=device, dtype=torch.float)
        logits, multi_logits = model(m_batch)
        
        probs = F.softmax(logits, dim=-1)
        
        out_list_binary=[]
        out_list_binary.extend(probs.tolist())
       
        target_list = ['bonafida' if i == 1 else 'spoof' for i in target.tolist()]
        label_list.extend(target_list)
        variant_list.extend(variant)
        random_token_list.extend(random_token)
        src_list.extend(src)
        file_name_list.extend(file_name)
        score_list_binary.extend(out_list_binary)
        
    end_time=time.time()
    print(f'{comment} use time : {end_time-start_time}s')
    with open(save_path, 'w') as file:
        for   score,v,f,r,s,l in zip(score_list_binary,variant_list,file_name_list,random_token_list,src_list,label_list):
            file.write('{} {} {} {} {} {} {}\n'.format(score[0],score[1],v,f,r,s,l))
'''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
'''    
get_score(eval_path='change this to the path to eval_list.txt',data_path='change this to VoiceWukong dataset path',save_path='change this to the path you want to save eval_score.txt',comment='en')
get_score(eval_path='change this to the path to zh_eval_list.txt',data_path='change this to VoiceWukong dataset path',save_path='change this to the path you want to save zh_eval_score.txt',comment='zh')